# 01. 베이스 모델 비교 (학습 전)

파인튜닝 전에 후보 모델들이 **test 세트(83문항)** 를 얼마나 잘 푸는지 비교합니다.
결과는 베이스 모델 선택 근거이자, 나중에 파인튜닝 효과를 보여 줄 **기준선**이 됩니다.

**시작 전**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택

평가 기준은 QA 검수(`pipeline/filter_qa.py`)와 같은 규칙을 씁니다 (`evaluation/metrics.py`).
- `uses_gwain`, `hao_ratio`, `no_modern_polite`, `no_hanja`: 페르소나 말투
- `years_correct`: 답변 속 연도가 사료와 맞는지
- `abstain_correct`: 답이 없는 질문엔 모른다고, 답이 있는 질문엔 답하는지
- `rouge_l`: 정답 답변과 글자 단위 유사도
- `tok_per_char`: 한국어 토큰 효율 (낮을수록 같은 글을 적은 토큰으로 처리 → CPU에서 빠름)

In [ ]:
# 1. 저장소 받기
!git clone -q https://github.com/leejinwo0/sejong-gpt.git 2>/dev/null || (cd sejong-gpt && git pull -q)
%cd /content/sejong-gpt
!ls data/dataset

In [ ]:
# 2. 패키지 (Colab 기본 설치본을 최신으로)
!pip install -q -U transformers accelerate

In [ ]:
# 3. (선택) Hugging Face 로그인 — 접근 동의가 필요한 모델이 있을 때만
# from huggingface_hub import login; login()

In [ ]:
# 4. 후보 모델과 test 세트
import json, time, gc
import torch
from pathlib import Path

CANDIDATES = {
    "exaone-3.5-2.4b": "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct",
    "exaone-4.0-1.2b": "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "kanana-1.5-2.1b": "kakaocorp/kanana-1.5-2.1b-instruct-2505",
    "hyperclovax-1.5b": "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
}
# 더 최신 소형 모델이 있으면 여기에 추가하세요. 각 모델의 라이선스(상업적 이용 가능 여부)도 모델 페이지에서 확인.

test_rows = [json.loads(l) for l in open("data/dataset/test.jsonl", encoding="utf-8")]
chunks = {c["chunk_id"]: c for c in map(json.loads, open("data/chunks/chunks.jsonl", encoding="utf-8"))}
print(f"test {len(test_rows)}문항, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음'}")

OUT_DIR = Path("evaluation/results/base"); OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 5. 생성 함수
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_NEW_TOKENS = 300

def to_prompt(tokenizer, messages):
    """system 역할을 지원하지 않는 템플릿이면 system을 user 앞에 붙여서 처리."""
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        merged = [{"role": "user", "content": messages[0]["content"] + "\n\n" + messages[1]["content"]}]
        return tokenizer.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)

def run_model(name, repo):
    tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        repo, torch_dtype=torch.float16, device_map="cuda", trust_remote_code=True)
    model.eval()
    outputs = []
    for i, row in enumerate(test_rows):
        prompt = to_prompt(tokenizer, row["messages"][:-1])   # 정답(assistant) 제외
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        start = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        outputs.append({
            "id": row["id"], "type": row["type"], "pred": pred,
            "ref": row["messages"][-1]["content"],
            "context_chunk_ids": row["context_chunk_ids"],
            "input_tokens": inputs["input_ids"].shape[1], "input_chars": len(prompt),
            "seconds": round(time.time() - start, 2),
        })
        if i % 20 == 0:
            print(f"  [{name}] {i+1}/{len(test_rows)}: {pred[:60]}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return outputs

In [ ]:
# 6. 모델별 실행 (모델당 5~15분). 이미 결과가 있는 모델은 건너뜀
for name, repo in CANDIDATES.items():
    path = OUT_DIR / f"{name}.jsonl"
    if path.exists():
        print(f"{name}: 결과 있음, 건너뜀"); continue
    print(f"=== {name} ({repo}) ===")
    try:
        outs = run_model(name, repo)
    except Exception as e:
        print(f"  실패: {e}"); gc.collect(); torch.cuda.empty_cache(); continue
    with open(path, "w", encoding="utf-8") as f:
        for o in outs:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")

In [ ]:
# 7. 점수 계산과 비교표
import pandas as pd
from evaluation.metrics import score_one, summarize

summary_rows, all_summaries = [], {}
for path in sorted(OUT_DIR.glob("*.jsonl")):
    outs = [json.loads(l) for l in open(path, encoding="utf-8")]
    scored = []
    for o in outs:
        ctx = "\n".join(chunks[c]["text"] for c in o["context_chunk_ids"])
        scored.append({"type": o["type"], "scores": score_one(o["pred"], o["ref"], o["type"], ctx)})
    s = summarize(scored)
    all_summaries[path.stem] = s
    ov = s["overall"]
    summary_rows.append({
        "model": path.stem,
        "과인 사용": ov.get("uses_gwain"), "하오체 비율": ov.get("hao_ratio"),
        "현대존댓말 없음": ov.get("no_modern_polite"), "한자 없음": ov.get("no_hanja"),
        "연도 정확": ov.get("years_correct"), "모름 판단": ov.get("abstain_correct"),
        "ROUGE-L": ov.get("rouge_l"), "평균 길이": ov.get("length"),
        "토큰/글자": round(sum(o["input_tokens"] for o in outs) / sum(o["input_chars"] for o in outs), 3),
        "초/문항(GPU)": round(sum(o["seconds"] for o in outs) / len(outs), 2),
    })

table = pd.DataFrame(summary_rows).set_index("model")
json.dump(all_summaries, open(OUT_DIR / "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
table

In [ ]:
# 8. 같은 질문에 대한 모델별 답변 비교 (눈으로 확인)
outs_by_model = {p.stem: {o["id"]: o for o in map(json.loads, open(p, encoding="utf-8"))} for p in sorted(OUT_DIR.glob("*.jsonl"))}
sample_ids = [r["id"] for r in test_rows if r["type"] in ("fact", "unanswerable")][:2] + \
             [r["id"] for r in test_rows if r["type"].startswith("persona_")][:2]
for sid in sample_ids:
    row = next(r for r in test_rows if r["id"] == sid)
    print("=" * 80)
    print(f"[{row['type']}] Q:", row["messages"][1]["content"].split("[질문]")[-1].strip())
    print("정답:", row["messages"][-1]["content"][:150])
    for m, outs in outs_by_model.items():
        print(f"- {m}: {outs[sid]['pred'][:150]}")

## 결과 저장

아래 셀로 결과를 내려받아 로컬 저장소의 `evaluation/results/base/`에 넣고 커밋하세요.
파인튜닝 후 같은 test 세트로 다시 평가해서 이 표와 비교합니다.

In [ ]:
!cd evaluation/results && zip -qr base_results.zip base
from google.colab import files
files.download("evaluation/results/base_results.zip")